In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import XLMRobertaTokenizer, XLMRobertaForMaskedLM
from torch.utils.data import Dataset, DataLoader
from transformers.optimization import AdamW
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm

In [2]:
PARAMS = {}
PARAMS['max_length'] = 128
PARAMS['pad_to_max_length'] = True
PARAMS['return_attention_mask'] = True
PARAMS['truncation_strategy'] = 'longest_first'
PARAMS['add_special_tokens'] = True
PARAMS['do_lower_case'] = False
TOKENIZER = XLMRobertaTokenizer.from_pretrained('xlm-roberta-large', do_lower_case=False)

In [3]:
MODEL = XLMRobertaForMaskedLM.from_pretrained('xlm-roberta-large')
MODEL = MODEL.to('cuda:0')

In [4]:
OPTIM = AdamW(MODEL.parameters(), lr=1e-5)

In [5]:
BSIZE = 4

In [6]:
def preProcess(inputs):
    np.random.seed(2017)
    mask = np.random.rand(*inputs.shape) < 0.15 
    mask[inputs<=2] = False
    labels =  -1 * np.ones(mask.shape, dtype=int)
    labels[mask] = inputs[mask]
    mlm = np.copy(inputs)
    mask2 = mask  & (np.random.rand(*inputs.shape)<0.90)
    mlm[mask2] = 250001
    random2 = mask2  & (np.random.rand(*inputs.shape) < 1/9)
    mlm[random2] = np.random.randint(3, 250001, random2.sum())
    return mlm, labels

def tokenizeText(text):
    text = str(text) + ' '
    if len(text.split()) >= 150:
        text = text.split()
        text = text[:150]
        text = ' '.join(text)
    text = TOKENIZER.encode_plus(text, **PARAMS)
    tokens = text['input_ids']
    attens = text['attention_mask']
    return tokens, attens

In [12]:
class ModelData(Dataset):
    
    def __init__(self, mode):
        train = pd.read_csv('../../data/process/pseudo/train_combine.csv', nrows=100)
        valid = pd.read_csv('../../data/process/pseudo/valid_combine.csv', nrows=100)
        test = pd.read_csv('../../data/process/pseudo/test_combine.csv', nrows=100)
        self.data = train[['comment_text']]
        self.data = self.data.append(valid[['comment_text']])
        self.data = self.data.append(test[['comment_text']])
        self.data = self.data.sample(frac=1., random_state=2017).reset_index(drop=True)
        self.mode = mode
        return None
    
    def __len__(self):
        return 100
    
    def __getitem__(self, idx):
        idx = np.random.randint(0,len(self.data))
        text = self.data.loc[idx, 'comment_text']
        tokens = np.array(tokenizeText(text.strip())[0])
        mlm, labels = preProcess(tokens)
        data = {}
        data['inputs'] = mlm
        data['labels'] = labels
        return data

In [13]:
data = ModelData('train')
data = DataLoader(data, batch_size=BSIZE , drop_last=True, num_workers=3)

In [14]:
def saveModel(model, epoch, loss):
    results = {}
    results['model_state_dict'] = model.state_dict()
    results['loss'] = loss
    results['metric'] = metric
    results['epoch'] = epoch
    torch.save(results, '../../model/pretrain/model.pt')
    return None

def trainModel():
    logfile = '../../model/pretrain/logfile.txt'
    if os.path.exists(logfile): os.remove(logfile)
    logfile = open(logfile, 'w', buffering=1)
    MODEL.train()
    checkpoint = np.NINF
    for epoch in range(20):
        step = 0
        OPTIM.zero_grad()
        losses = []
        tq = tqdm(total=len(data) * BSIZE, disable=False)
        for sample in data:
            step += 1
            labels = sample['labels'].long().to('cuda:0')
            inputs = sample['inputs'].long().to('cuda:0')
            print(label, inputs)
            preds = MODEL(input_ids=inputs, masked_lm_labels=labels)
            print(preds[0])
            loss = preds[0].to('cuda:0')
            loss.backward()
            if step % 4 == 0:
                step = 0
                clip_grad_norm_(model.parameters(), 1.)
                OPTIM.step()
                OPTIM.zero_grad()
            losses.append(loss.cpu().data.item())
            train_loss = round(np.mean(losses),4)
            tq.update(BSIZE)
            tq.set_postfix(train_loss='{:.4f}'.format(train_loss))
        del sample, label, preds, inputs
        torch.cuda.empty_cache()
        saveModel(MODEL, epoch, train_loss)
        tq.close()
        logtext  = 'Epoch - {} | '.format(epoch)
        logtext += 'Loss - {:.4f} | '.format(train_loss)
        logtext += '\n'
        logfile.write(logtext)
        sys.stdout.flush()
    logfile.close()
    return None

In [15]:
trainModel()


  0%|          | 0/100 [00:00<?, ?it/s]

RuntimeError: CUDA error: device-side assert triggered

In [16]:
for sample in data:
    print(sample)

{'inputs': tensor([[     0,  22010,   3427, 250001,  78874,     34, 250001,    187,    275,
            277, 250001,      2,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1, 

In [18]:
sample['inputs'].shape

torch.Size([4, 128])

In [19]:
sample['labels'].shape

torch.Size([4, 128])

In [20]:
sample['labels']

tensor([[    -1,     -1,     -1,    444,     -1,     -1,   6097,     -1,     -1,
             -1,     70,     -1,     -1,     -1,     -1,     -1,     -1,     -1,
           5947,     -1,    581,     -1,     -1,     -1,     -1,     -1,     -1,
             58,     -1,     -1,   4568,  14277,     -1,     -1,     -1,     -1,
             -1,  46512,     -1,     -1,     -1,     -1,     -1,     -1,      7,
             -1,     -1,     -1,     -1,     -1,     -1,     -1,     -1,     -1,
             -1,     -1,     -1,     -1,     -1,     -1,     -1,     -1,     -1,
             -1,     -1,  15490,     -1,     -1,     -1,     -1,    538,     -1,
             -1,     -1,     -1,   1295,     -1,     -1,     -1,     -1,     10,
             -1,     -1, 127298,     -1,     -1,     -1,     -1,  54452,     -1,
             -1,     -1,     -1,     -1,     -1,     -1,     -1,     -1,     -1,
             -1,     -1, 177229,     -1,     -1,     -1,     -1,     -1,     -1,
             -1,     -1,    

In [21]:
sample['inputs']

tensor([[     0,     44,    568, 250001,  47644,  33233, 250001, 124519,     12,
           2161, 250001,  51521,   4165,      4,     44,     58,  44072, 102057,
         250001,     12, 250001,   5533,    164,  63127,    297,  74831,     58,
         250001,     83,     70, 250001, 250001,     47,    186,  59121,    678,
             10, 250001,      5,     44,     58,  44072, 102057,  83479,      7,
             58,     58,     83,     70,   5117,  14277,     47,    186,  59121,
          15490,   1632,      4,    136,    756, 214493,    214,  72304,    765,
           2809,  59121, 250001,     10,  46512,      5,  86120, 250001,      4,
             70,  44930,  15549, 250001,  17368,     47,    959,  17368,  84988,
          46512,    509, 250001, 173969,    297,    678,     70, 250001,    111,
           4612, 102057,  83479,      7,    136,   8162,   5608,  44961,    903,
              5, 116267, 250001,     83,    450,   2174,     70,  44759,    111,
            903,   5582,    

In [22]:
MODEL

XLMRobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm(torch.Size([1024]), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm(torch.Size([1024

In [23]:
MODEL(sample['inputs'].long())

RuntimeError: CUDA error: device-side assert triggered

In [25]:
x = sample['inputs'].long()

In [31]:
x

torch.int64

In [34]:
torch.tensor(x).cuda()

/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  """Entry point for launching an IPython kernel.


RuntimeError: CUDA error: device-side assert triggered